In [8]:
# Cargar Bronze desde Supabase y preparar Silver/Gold
import sys
import os

import pandas as pd

sys.path.insert(0, os.path.abspath("../.secrets"))
from db_config import get_connection

conn = get_connection()
print("Conectado a Supabase")

tablas = [
    "dim_asesor",
    "fact_reclutamiento",
    "fact_rendimiento_mensual",
    "fact_capacitacion",
    "fact_calidad",
    "fact_incidencias",
    "fact_adherencia",
    "fact_clima",
]

data_bronze = {}
for t in tablas:
    data_bronze[t] = pd.read_sql(f"SELECT * FROM bronze.{t}", conn)

print("Tablas bronze cargadas:", list(data_bronze.keys()))

Conectado a Supabase


C:\Users\Asus\AppData\Local\Temp\ipykernel_3896\1750126979.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data_bronze[t] = pd.read_sql(f"SELECT * FROM bronze.{t}", conn)


Tablas bronze cargadas: ['dim_asesor', 'fact_reclutamiento', 'fact_rendimiento_mensual', 'fact_capacitacion', 'fact_calidad', 'fact_incidencias', 'fact_adherencia', 'fact_clima']


## Capa Silver
Limpieza: tipos, duplicados, nulos y normalización de textos.

In [9]:
import re

def _normalize_colname(name: str) -> str:
    s = str(name).strip().lower()
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^a-z0-9_]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [_normalize_colname(c) for c in out.columns]
    return out


def strip_strings(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.select_dtypes(include=["object", "string"]).columns:
        out[c] = (
            out[c]
            .astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .replace({"nan": pd.NA, "": pd.NA, "none": pd.NA, "null": pd.NA})
        )
    return out


def coerce_datetime(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = pd.to_datetime(out[c], errors="coerce")
    return out


def coerce_numeric(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    return out


def drop_critical_nulls(df: pd.DataFrame, critical_cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    present = [c for c in critical_cols if c in out.columns]
    if present:
        out = out.dropna(subset=present)
    return out


def invalidate_negatives(df: pd.DataFrame, non_negative_cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    for c in non_negative_cols:
        if c in out.columns:
            out.loc[out[c] < 0, c] = pd.NA
    return out


def silver_rendimiento_mensual(df: pd.DataFrame) -> pd.DataFrame:
    out = normalize_columns(df)
    out = out.drop_duplicates()
    out = strip_strings(out)

    out = coerce_datetime(out, ["fecha", "fecha_mes", "mes", "periodo"])

    numeric_candidates = [
        "tmo",
        "tmo_promedio",
        "tmo_segundos",
        "aht",
        "aht_segundos",
        "llamadas",
        "contactos",
        "atenciones",
        "ventas",
        "cumplimiento",
    ]
    out = coerce_numeric(out, [c for c in numeric_candidates if c in out.columns])
    out = invalidate_negatives(out, [c for c in numeric_candidates if c in out.columns])

    out = drop_critical_nulls(out, ["id_asesor"])
    return out


def silver_calidad(df: pd.DataFrame) -> pd.DataFrame:
    out = normalize_columns(df)
    out = out.drop_duplicates()
    out = strip_strings(out)

    out = coerce_datetime(out, ["fecha", "fecha_evaluacion", "periodo", "mes"])

    numeric_candidates = ["score", "calidad", "nota", "puntaje", "evaluacion", "n_evaluacion"]
    out = coerce_numeric(out, [c for c in numeric_candidates if c in out.columns])
    out = invalidate_negatives(out, [c for c in numeric_candidates if c in out.columns])

    if "score" not in out.columns:
        for alt in ["calidad", "nota", "puntaje", "evaluacion"]:
            if alt in out.columns:
                out = out.rename(columns={alt: "score"})
                break

    out = drop_critical_nulls(out, ["id_asesor", "score"])
    return out


def silver_incidencias(df: pd.DataFrame) -> pd.DataFrame:
    out = normalize_columns(df)
    out = out.drop_duplicates()
    out = strip_strings(out)

    out = coerce_datetime(out, ["fecha", "fecha_incidencia", "periodo", "mes"])

    numeric_candidates = ["incidencias", "cantidad", "total", "n_incidencias"]
    out = coerce_numeric(out, [c for c in numeric_candidates if c in out.columns])
    out = invalidate_negatives(out, [c for c in numeric_candidates if c in out.columns])

    if "incidencias" not in out.columns:
        for alt in ["cantidad", "total", "n_incidencias"]:
            if alt in out.columns:
                out = out.rename(columns={alt: "incidencias"})
                break

    out = drop_critical_nulls(out, ["id_asesor"])
    return out


def silver_adherencia(df: pd.DataFrame) -> pd.DataFrame:
    out = normalize_columns(df)
    out = out.drop_duplicates()
    out = strip_strings(out)

    out = coerce_datetime(out, ["fecha", "periodo", "mes"])

    numeric_candidates = ["adherencia", "porcentaje_adherencia", "pct_adherencia"]
    out = coerce_numeric(out, [c for c in numeric_candidates if c in out.columns])
    out = invalidate_negatives(out, [c for c in numeric_candidates if c in out.columns])

    # Normalizar a proporción 0-1 si viene 0-100
    for c in ["adherencia", "porcentaje_adherencia", "pct_adherencia"]:
        if c in out.columns:
            out.loc[out[c] > 1, c] = out.loc[out[c] > 1, c] / 100

    if "adherencia" not in out.columns:
        for alt in ["porcentaje_adherencia", "pct_adherencia"]:
            if alt in out.columns:
                out = out.rename(columns={alt: "adherencia"})
                break

    out = drop_critical_nulls(out, ["id_asesor", "adherencia"])
    return out


def silver_capacitacion(df: pd.DataFrame) -> pd.DataFrame:
    out = normalize_columns(df)
    out = out.drop_duplicates()
    out = strip_strings(out)

    out = coerce_datetime(out, ["fecha", "fecha_capacitacion", "periodo", "mes"])

    numeric_candidates = ["horas", "duracion_horas", "duracion", "score", "nota"]
    out = coerce_numeric(out, [c for c in numeric_candidates if c in out.columns])
    out = invalidate_negatives(out, [c for c in numeric_candidates if c in out.columns])

    out = drop_critical_nulls(out, ["id_asesor"])
    return out


def silver_passthrough(df: pd.DataFrame) -> pd.DataFrame:
    """Para tablas no priorizadas: limpieza mínima y consistente."""
    out = normalize_columns(df)
    out = out.drop_duplicates()
    out = strip_strings(out)
    return out


# Construcción Silver manteniendo las mismas keys que Bronze
SILVER_BUILDERS = {
    "fact_rendimiento_mensual": silver_rendimiento_mensual,
    "fact_calidad": silver_calidad,
    "fact_incidencias": silver_incidencias,
    "fact_adherencia": silver_adherencia,
    "fact_capacitacion": silver_capacitacion,
}

data_silver = {}
for nombre, df in data_bronze.items():
    builder = SILVER_BUILDERS.get(nombre, silver_passthrough)
    data_silver[nombre] = builder(df)

print("Silver generado:", list(data_silver.keys()))

Silver generado: ['dim_asesor', 'fact_reclutamiento', 'fact_rendimiento_mensual', 'fact_capacitacion', 'fact_calidad', 'fact_incidencias', 'fact_adherencia', 'fact_clima']


## Capa Gold
Agregados de negocio: resumen por asesor y métricas consolidadas.

In [10]:
# Gold: dimensión asesor (limpia, lista para reporting)
gold_dim_asesor = data_silver["dim_asesor"].copy()


def _pick_first_existing(df: pd.DataFrame, candidates: list[str]) -> str | None:
    for c in candidates:
        if c in df.columns:
            return c
    return None


def _safe_divide(numer: pd.Series, denom: pd.Series) -> pd.Series:
    denom = denom.replace({0: pd.NA})
    out = numer / denom
    return out.replace([pd.NA, float("inf"), float("-inf")], pd.NA)


def _minmax_0_1(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    mn = s.min(skipna=True)
    mx = s.max(skipna=True)
    if pd.isna(mn) or pd.isna(mx) or mn == mx:
        return pd.Series([0.0] * len(s), index=s.index, dtype="float64")
    return ((s - mn) / (mx - mn)).astype("float64")


def build_gold_resumen_asesor(data_silver: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Construye un dataset Gold consolidado por id_asesor.

    Incluye KPIs de negocio, features para ML y una variable objetivo (riesgo)
    calculada con reglas basadas en percentiles.
    """

    base = data_silver.get("dim_asesor")
    if base is None or base.empty or "id_asesor" not in base.columns:
        raise ValueError("`dim_asesor` no existe o no contiene `id_asesor` en data_silver")

    gold = base[["id_asesor"]].drop_duplicates().copy()

    # -----------------
    # Rendimiento mensual
    # -----------------
    rm = data_silver.get("fact_rendimiento_mensual")
    if rm is not None and not rm.empty and "id_asesor" in rm.columns:
        llamadas_col = _pick_first_existing(rm, ["llamadas", "contactos", "atenciones"])
        duracion_col = _pick_first_existing(rm, ["duracion_total", "duracion", "aht_segundos", "aht", "tmo_segundos", "tmo"])
        tmo_col = _pick_first_existing(rm, ["tmo", "tmo_promedio", "tmo_segundos", "aht", "aht_segundos"])

        rm_num = rm.copy()
        for c in [llamadas_col, duracion_col, tmo_col]:
            if c and c in rm_num.columns:
                rm_num[c] = pd.to_numeric(rm_num[c], errors="coerce")

        agg = rm_num.groupby("id_asesor", as_index=False).agg(
            total_llamadas=(llamadas_col, "sum") if llamadas_col else ("id_asesor", "size"),
            duracion_total=(duracion_col, "sum") if duracion_col else ("id_asesor", "size"),
            tmo_promedio=(tmo_col, "mean") if tmo_col else ("id_asesor", "size"),
        )

        # si no había columna de duración/tmo real, evitar basura: set NA
        if not duracion_col:
            agg["duracion_total"] = pd.NA
        if not tmo_col:
            agg["tmo_promedio"] = pd.NA

        gold = gold.merge(agg, on="id_asesor", how="left")
    else:
        gold[["total_llamadas", "duracion_total", "tmo_promedio"]] = pd.NA

    # --------
    # Calidad
    # --------
    cal = data_silver.get("fact_calidad")
    if cal is not None and not cal.empty and "id_asesor" in cal.columns:
        score_col = _pick_first_existing(cal, ["score", "calidad", "nota", "puntaje", "evaluacion"])
        cal_num = cal.copy()
        if score_col:
            cal_num[score_col] = pd.to_numeric(cal_num[score_col], errors="coerce")

        agg = cal_num.groupby("id_asesor", as_index=False).agg(
            score_calidad_promedio=(score_col, "mean") if score_col else ("id_asesor", "size"),
        )
        if not score_col:
            agg["score_calidad_promedio"] = pd.NA
        gold = gold.merge(agg, on="id_asesor", how="left")
    else:
        gold["score_calidad_promedio"] = pd.NA

    # ------------
    # Incidencias
    # ------------
    inc = data_silver.get("fact_incidencias")
    if inc is not None and not inc.empty and "id_asesor" in inc.columns:
        inc_col = _pick_first_existing(inc, ["incidencias", "cantidad", "total", "n_incidencias"])
        inc_num = inc.copy()
        if inc_col:
            inc_num[inc_col] = pd.to_numeric(inc_num[inc_col], errors="coerce")

        agg = inc_num.groupby("id_asesor", as_index=False).agg(
            total_incidencias=(inc_col, "sum") if inc_col else ("id_asesor", "size"),
        )
        if not inc_col:
            agg["total_incidencias"] = pd.NA
        gold = gold.merge(agg, on="id_asesor", how="left")
    else:
        gold["total_incidencias"] = pd.NA

    # -----------
    # Adherencia
    # -----------
    adh = data_silver.get("fact_adherencia")
    if adh is not None and not adh.empty and "id_asesor" in adh.columns:
        adh_col = _pick_first_existing(adh, ["adherencia", "porcentaje_adherencia", "pct_adherencia"])
        adh_num = adh.copy()
        if adh_col:
            adh_num[adh_col] = pd.to_numeric(adh_num[adh_col], errors="coerce")

        agg = adh_num.groupby("id_asesor", as_index=False).agg(
            adherencia_promedio=(adh_col, "mean") if adh_col else ("id_asesor", "size"),
        )
        if not adh_col:
            agg["adherencia_promedio"] = pd.NA
        gold = gold.merge(agg, on="id_asesor", how="left")
    else:
        gold["adherencia_promedio"] = pd.NA

    # --------------
    # Capacitaciones
    # --------------
    cap = data_silver.get("fact_capacitacion")
    if cap is not None and not cap.empty and "id_asesor" in cap.columns:
        # KPI pedido: total_capacitaciones (conteo de eventos)
        agg = cap.groupby("id_asesor", as_index=False).size().rename(columns={"size": "total_capacitaciones"})
        gold = gold.merge(agg, on="id_asesor", how="left")
    else:
        gold["total_capacitaciones"] = pd.NA

    # -----------------
    # Features para ML
    # -----------------
    gold["total_llamadas"] = pd.to_numeric(gold["total_llamadas"], errors="coerce").fillna(0)
    gold["total_incidencias"] = pd.to_numeric(gold["total_incidencias"], errors="coerce").fillna(0)

    gold["ratio_incidencias"] = _safe_divide(gold["total_incidencias"], gold["total_llamadas"]).fillna(0.0)

    # Normalizaciones (0-1) para uso en modelo / dashboards
    gold["tmo_normalizado"] = _minmax_0_1(gold["tmo_promedio"])
    gold["calidad_normalizada"] = _minmax_0_1(gold["score_calidad_promedio"])
    gold["adherencia_normalizada"] = _minmax_0_1(gold["adherencia_promedio"])

    # ---------------
    # Target: riesgo
    # ---------------
    tmo_p75 = pd.to_numeric(gold["tmo_promedio"], errors="coerce").quantile(0.75)
    cal_p25 = pd.to_numeric(gold["score_calidad_promedio"], errors="coerce").quantile(0.25)
    inc_p75 = pd.to_numeric(gold["ratio_incidencias"], errors="coerce").quantile(0.75)

    cond_tmo = pd.to_numeric(gold["tmo_promedio"], errors="coerce") >= tmo_p75
    cond_cal = pd.to_numeric(gold["score_calidad_promedio"], errors="coerce") <= cal_p25
    cond_inc = pd.to_numeric(gold["ratio_incidencias"], errors="coerce") >= inc_p75

    gold["riesgo"] = (cond_tmo & cond_cal & cond_inc).astype("int64")

    # Limpieza final de nulos en KPIs clave (para Looker/ML)
    for c in [
        "duracion_total",
        "tmo_promedio",
        "score_calidad_promedio",
        "adherencia_promedio",
        "total_capacitaciones",
    ]:
        if c in gold.columns:
            gold[c] = pd.to_numeric(gold[c], errors="coerce")

    # Orden de columnas (sin romper si faltan)
    ordered = [
        "id_asesor",
        "total_llamadas",
        "duracion_total",
        "tmo_promedio",
        "score_calidad_promedio",
        "total_incidencias",
        "adherencia_promedio",
        "total_capacitaciones",
        "ratio_incidencias",
        "tmo_normalizado",
        "calidad_normalizada",
        "adherencia_normalizada",
        "riesgo",
    ]
    cols = [c for c in ordered if c in gold.columns] + [c for c in gold.columns if c not in ordered]
    return gold[cols]


gold_resumen_asesor = build_gold_resumen_asesor(data_silver)

data_gold = {
    "dim_asesor": gold_dim_asesor,
    "resumen_por_asesor": gold_resumen_asesor,
}
print("Gold generado:", list(data_gold.keys()))

Gold generado: ['dim_asesor', 'resumen_por_asesor']


## Crear schemas y subir Silver y Gold a Supabase

In [11]:
from sqlalchemy import create_engine

# URI desde la misma config (ajusta si usas variables de entorno)
import os
from dotenv import load_dotenv
load_dotenv(os.path.abspath("../.env"))
if not os.path.exists(os.path.abspath("../.env")):
    load_dotenv()

user = os.getenv("DB_USER", "postgres.zuxjvpqgdpprzostfiuf")
password = os.getenv("DB_PASSWORD", "xmatansa123")
host = os.getenv("DB_HOST", "aws-0-us-west-2.pooler.supabase.com")
port = os.getenv("DB_PORT", "5432")
db = os.getenv("DB_NAME", "postgres")
uri = f"postgresql://{user}:{password}@{host}:{port}/{db}"
engine = create_engine(uri)

# Crear schemas vía SQLAlchemy (más resiliente a cortes de conn psycopg2)
from sqlalchemy import text

with engine.begin() as con:
    con.execute(text("CREATE SCHEMA IF NOT EXISTS silver;"))
    con.execute(text("CREATE SCHEMA IF NOT EXISTS gold;"))

print("Schemas silver y gold creados (o ya existían).")

OperationalError: server closed the connection unexpectedly
	This probably means the server terminated abnormally
	before or while processing the request.
server closed the connection unexpectedly
	This probably means the server terminated abnormally
	before or while processing the request.


In [7]:
from sqlalchemy import text
from sqlalchemy.exc import SQLAlchemyError


def safe_overwrite_table(df: pd.DataFrame, engine, schema: str, table: str) -> None:
    """Sobrescribe datos sin dropear la tabla (evita romper vistas dependientes).

    Estrategia:
    - Si la tabla no existe: create via to_sql(replace)
    - Si existe: TRUNCATE + to_sql(append)

    Nota: si cambian columnas, versiona la tabla (ej: *_v2) y migra vistas.
    """
    try:
        with engine.connect() as con:
            exists = con.execute(
                text(
                    """
                    SELECT 1
                    FROM information_schema.tables
                    WHERE table_schema = :schema AND table_name = :table
                    LIMIT 1
                    """
                ),
                {"schema": schema, "table": table},
            ).scalar()

        if not exists:
            df.to_sql(table, engine, schema=schema, if_exists="replace", index=False)
            return

        with engine.begin() as con:
            con.execute(text(f'TRUNCATE TABLE "{schema}"."{table}";'))

        df.to_sql(table, engine, schema=schema, if_exists="append", index=False)

    except SQLAlchemyError as e:
        raise RuntimeError(f"Error subiendo {schema}.{table}: {e}")


# Subir tablas Silver
for nombre, df in data_silver.items():
    safe_overwrite_table(df, engine, schema="silver", table=nombre)
    print(f"Silver: {nombre} -> silver.{nombre}")

# Subir tablas Gold
for nombre, df in data_gold.items():
    safe_overwrite_table(df, engine, schema="gold", table=nombre)
    print(f"Gold: {nombre} -> gold.{nombre}")

print("\nSilver y Gold subidos a Supabase correctamente.")

Silver: dim_asesor -> silver.dim_asesor
Silver: fact_reclutamiento -> silver.fact_reclutamiento
Silver: fact_rendimiento_mensual -> silver.fact_rendimiento_mensual
Silver: fact_capacitacion -> silver.fact_capacitacion
Silver: fact_calidad -> silver.fact_calidad
Silver: fact_incidencias -> silver.fact_incidencias
Silver: fact_adherencia -> silver.fact_adherencia
Silver: fact_clima -> silver.fact_clima
Gold: dim_asesor -> gold.dim_asesor
Gold: resumen_por_asesor -> gold.resumen_por_asesor

Silver y Gold subidos a Supabase correctamente.
